In [ ]:
import os
import glob
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import mne
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score
import warnings

# Suppress verbose processing logs
warnings.filterwarnings('ignore')
mne.set_log_level('WARNING')

DEVICE = torch.device("cuda:1" if torch.cuda.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
print(f"Executing pipeline on device: {DEVICE}")

# --- Configuration ---
# 🌟 UPDATED: Using absolute paths so it always finds your existing files regardless of where the notebook is run
BASE_DIR = "/home/gella.saikrishna/code/Learning-with-FrameProjections"
LOCAL_DATA_DIR = "/home/gella.saikrishna/sleep-edf-database-expanded-1.0.0/sleep-edf-database-expanded-1.0.0/sleep-cassette"
SAVE_DIR = os.path.join(BASE_DIR, "data")
OUTPUT_FILE = os.path.join(SAVE_DIR, "sleep_combined.pt")
CHECKPOINT_DIR = os.path.join(BASE_DIR, "checkpoints")

os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

EPOCH_SEC = 30       
TARGET_SFREQ = 100   
TIME_STEPS = 3000    
LATENT_DIM = 128     
NUM_CLASSES = 5      
PRETRAIN_EPOCHS = 40
EVAL_EPOCHS = 30
BATCH_SIZE = 64

STAGE_MAPPING = {
    "Sleep stage W": 0, "Sleep stage 1": 1, "Sleep stage 2": 2,
    "Sleep stage 3": 3, "Sleep stage 4": 3, "Sleep stage R": 4
}

# ==========================================
# 0. DATA PREPROCESSING (Auto-build if missing)
# ==========================================
if not os.path.exists(OUTPUT_FILE):
    print(f"\n⚠️ Dataset not found at {OUTPUT_FILE}. Starting preprocessing from raw EDF files...")
    psg_files = sorted(glob.glob(os.path.join(LOCAL_DATA_DIR, "*PSG.edf")))
    hypno_files = sorted(glob.glob(os.path.join(LOCAL_DATA_DIR, "*Hypnogram.edf")))
    raw_data_files = list(zip(psg_files, hypno_files))

    if len(raw_data_files) == 0:
        raise FileNotFoundError(f"No EDF files found in {LOCAL_DATA_DIR}. Please check your path!")

    print(f"✅ Found {len(raw_data_files)} paired recording sessions locally.")
    
    all_epochs_list = []
    all_labels_list = []

    print("Preprocessing raw EEG signals (This may take a few minutes)...")
    for psg_file, hypno_file in raw_data_files:
        raw = mne.io.read_raw_edf(psg_file, preload=True)
        eeg_channels = [ch for ch in raw.ch_names if 'EEG' in ch]
        if len(eeg_channels) > 0:
            raw.pick_channels([eeg_channels[0]])
        else:
            continue
            
        raw.filter(l_freq=0.5, h_freq=30.0, fir_design='firwin')
        if raw.info['sfreq'] != TARGET_SFREQ:
            raw.resample(TARGET_SFREQ, npad="auto")
            
        annotations = mne.read_annotations(hypno_file)
        raw.set_annotations(annotations, emit_warning=False)
        
        events, event_id = mne.events_from_annotations(raw, event_id=STAGE_MAPPING, chunk_duration=float(EPOCH_SEC))
        tmax = 30.0 - 1.0 / TARGET_SFREQ 
        epochs = mne.Epochs(raw=raw, events=events, event_id=event_id, tmin=0.0, tmax=tmax, baseline=None, preload=True, reject=None)
        
        data = epochs.get_data() 
        labels = epochs.events[:, -1]
        data = np.squeeze(data, axis=1)
        
        all_epochs_list.append(data)
        all_labels_list.append(labels)

    X = np.concatenate(all_epochs_list, axis=0)
    y = np.concatenate(all_labels_list, axis=0)

    print("Applying global Z-score standardization...")
    X = (X - np.mean(X)) / (np.std(X) + 1e-6)

    print("Splitting dataset into stratified subsets...")
    X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
    X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.20, random_state=42, stratify=y_train_val)

    processed_dataset_object = {
        "train": {"samples": torch.from_numpy(X_train).float(), "labels": torch.from_numpy(y_train).long()},
        "val": {"samples": torch.from_numpy(X_val).float(), "labels": torch.from_numpy(y_val).long()},
        "test": {"samples": torch.from_numpy(X_test).float(), "labels": torch.from_numpy(y_test).long()}
    }
    torch.save(processed_dataset_object, OUTPUT_FILE)
    print(f"✅ Successfully created and saved custom splits to {OUTPUT_FILE}")
else:
    print(f"\n✅ Found existing preprocessed dataset at {OUTPUT_FILE}. Skipping raw extraction.")

# ==========================================
# 1. DATASETS (FOURIER REMOVED)
# ==========================================

class SleepEDF_Pretrain_Dataset(Dataset):
    def __init__(self, pt_file_path=OUTPUT_FILE, split="train"):
        data_obj = torch.load(pt_file_path, map_location="cpu")[split]
        self.data = data_obj["samples"]
        self.window = torch.hann_window(128)
        
    def __len__(self):
        return len(self.data)
        
    def __getitem__(self, idx):
        x_t = self.data[idx] 
        
        # 1. TIME BRANCH
        x_time = x_t.unsqueeze(0) 
        
        # [FOURIER BRANCH REMOVED]
        
        # 2. GABOR / WAVELET BRANCH
        x_stft = torch.stft(x_t, n_fft=128, hop_length=64, window=self.window, return_complex=True)
        x_wavelet = torch.abs(x_stft)[:64, :] 
        x_wavelet = F.pad(x_wavelet, (0, 1)) 
        x_wavelet = x_wavelet.unsqueeze(0) 
        
        # Now only returning 2 variables instead of 3
        return x_time, x_wavelet

class SleepEDF_Evaluation_Dataset(Dataset):
    def __init__(self, pt_file_path=OUTPUT_FILE, split="train"):
        data_obj = torch.load(pt_file_path, map_location="cpu")[split]
        # 🌟 UPDATED: Squeezing to dim 1 to maintain [N, 1, 3000] consistency and prevent transpose errors later
        self.samples = data_obj["samples"].unsqueeze(1) 
        self.labels = data_obj["labels"].long()

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx], self.labels[idx]

# ==========================================
# 2. NEURAL NETWORK BLOCKS
# ==========================================

class MyConv1dPadSame(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride, groups=1):
        super(MyConv1dPadSame, self).__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.kernel_size = kernel_size
        self.stride = stride
        self.groups = groups
        self.conv = torch.nn.Conv1d(in_channels=self.in_channels, out_channels=self.out_channels, 
                                    kernel_size=self.kernel_size, stride=self.stride, groups=self.groups)
    def forward(self, x):
        net = x
        in_dim = net.shape[-1]
        out_dim = (in_dim + self.stride - 1) // self.stride
        p = max(0, (out_dim - 1) * self.stride + self.kernel_size - in_dim)
        pad_left = p // 2
        pad_right = p - pad_left
        net = F.pad(net, (pad_left, pad_right), "constant", 0)
        return self.conv(net)

class MyMaxPool1dPadSame(nn.Module):
    def __init__(self, kernel_size):
        super(MyMaxPool1dPadSame, self).__init__()
        self.kernel_size = kernel_size
        self.stride = 1
        self.max_pool = torch.nn.MaxPool1d(kernel_size=self.kernel_size)
    def forward(self, x):
        net = x
        in_dim = net.shape[-1]
        out_dim = (in_dim + self.stride - 1) // self.stride
        p = max(0, (out_dim - 1) * self.stride + self.kernel_size - in_dim)
        pad_left = p // 2
        pad_right = p - pad_left
        net = F.pad(net, (pad_left, pad_right), "constant", 0)
        return self.max_pool(net)

class BasicBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride, groups, downsample, use_bn, use_do, is_first_block=False):
        super(BasicBlock, self).__init__()
        self.in_channels = in_channels
        self.kernel_size = kernel_size
        self.out_channels = out_channels
        self.stride = stride if downsample else 1
        self.groups = groups
        self.downsample = downsample
        self.is_first_block = is_first_block
        self.use_bn = use_bn
        self.use_do = use_do

        self.bn1 = nn.BatchNorm1d(in_channels)
        self.relu1 = nn.ReLU()
        self.do1 = nn.Dropout(p=0.5)
        self.conv1 = MyConv1dPadSame(in_channels=in_channels, out_channels=out_channels, kernel_size=kernel_size, stride=self.stride, groups=self.groups)

        self.bn2 = nn.BatchNorm1d(out_channels)
        self.relu2 = nn.ReLU()
        self.do2 = nn.Dropout(p=0.5)
        self.conv2 = MyConv1dPadSame(in_channels=out_channels, out_channels=out_channels, kernel_size=kernel_size, stride=1, groups=self.groups)
        self.max_pool = MyMaxPool1dPadSame(kernel_size=self.stride)

    def forward(self, x):
        identity = x
        out = x
        if not self.is_first_block:
            if self.use_bn: out = self.bn1(out)
            out = self.relu1(out)
            if self.use_do: out = self.do1(out)
        out = self.conv1(out)
        
        if self.use_bn: out = self.bn2(out)
        out = self.relu2(out)
        if self.use_do: out = self.do2(out)
        out = self.conv2(out)
        
        if self.downsample: identity = self.max_pool(identity)
        if self.out_channels != self.in_channels:
            identity = identity.transpose(-1,-2)
            ch1 = (self.out_channels-self.in_channels)//2
            ch2 = self.out_channels-self.in_channels-ch1
            identity = F.pad(identity, (ch1, ch2), "constant", 0)
            identity = identity.transpose(-1,-2)
        out += identity
        return out

def conbr_block_2d(in_channels, out_channels, kernel_size, stride, padding):
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size, stride=stride, padding=padding),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(inplace=True)
    )

class ResNet1D(nn.Module):
    def __init__(self, in_channels, base_filters, kernel_size, stride, groups, n_block, n_classes, downsample_gap=2, increasefilter_gap=4, use_bn=True, use_do=True, verbose=False, backbone=False, output_dim=200):
        super(ResNet1D, self).__init__()
        self.out_dim = output_dim
        self.backbone = backbone
        self.verbose = verbose
        self.n_block = n_block
        self.kernel_size = kernel_size
        self.stride = stride
        self.groups = groups
        self.use_bn = use_bn
        self.use_do = use_do
        self.downsample_gap = downsample_gap
        self.increasefilter_gap = increasefilter_gap

        self.first_block_conv = MyConv1dPadSame(in_channels=in_channels, out_channels=base_filters, kernel_size=self.kernel_size, stride=1)
        self.first_block_bn = nn.BatchNorm1d(base_filters)
        self.first_block_relu = nn.ReLU()
        out_channels = base_filters
                
        self.basicblock_list = nn.ModuleList()
        for i_block in range(self.n_block):
            is_first_block = True if i_block == 0 else False
            downsample = True if i_block % self.downsample_gap == 1 else False
            
            if is_first_block:
                in_channels = base_filters
                out_channels = in_channels
            else:
                in_channels = int(base_filters*2**((i_block-1)//self.increasefilter_gap))
                if (i_block % self.increasefilter_gap == 0) and (i_block != 0):
                    out_channels = in_channels * 2
                else:
                    out_channels = in_channels
            
            tmp_block = BasicBlock(in_channels=in_channels, out_channels=out_channels, kernel_size=self.kernel_size, stride = self.stride, groups = self.groups, downsample=downsample, use_bn = self.use_bn, use_do = self.use_do, is_first_block=is_first_block)
            self.basicblock_list.append(tmp_block)

        self.final_bn = nn.BatchNorm1d(out_channels)
        self.final_relu = nn.ReLU(inplace=True)
        self.dense = nn.Linear(out_channels, n_classes)
        self.dense2 = nn.Linear(out_channels, self.out_dim)
        
    def forward(self, x):
        # 🌟 FIX: Removed x = x.transpose(-1,-2) since Datasets already output correct [Batch, Channel, Length] shapes
        out = x
        
        out = self.first_block_conv(out)
        if self.use_bn: out = self.first_block_bn(out)
        out = self.first_block_relu(out)
        
        for i_block in range(self.n_block):
            net = self.basicblock_list[i_block]
            out = net(out)

        if self.use_bn: out = self.final_bn(out)
        out = self.final_relu(out)
        out = out.mean(-1)
        
        if self.backbone:
            out = self.dense2(out)
            return None, out
            
        out_class = self.dense(out)
        return out_class, out    

class UNET_2D_simp(nn.Module):
    def __init__(self, input_channels, output_channels, layer_n, spect_freq, spect_time, kernel_size):
        super(UNET_2D_simp, self).__init__()
        self.input_channels = input_channels
        self.layer_n = layer_n
        self.kernel_size = kernel_size
        self.output_channels = output_channels
        self.spec_freq = spect_freq
        self.spec_time = spect_time

        self.AvgPool2D1 = nn.AvgPool2d(kernel_size=(2, 2), stride=(2, 2))
        self.AvgPool2D2 = nn.AvgPool2d(kernel_size=(4, 4), stride=(4, 4))
        
        self.layer1 = self.down_layer_2d(self.input_channels, self.layer_n, self.kernel_size, stride=1, padding=self.kernel_size//2)
        self.layer2 = self.down_layer_2d(self.layer_n, self.layer_n * 2, self.kernel_size, stride=2, padding=self.kernel_size//2)
        self.layer3 = self.down_layer_2d(self.layer_n * 2 + self.input_channels, self.layer_n * 3, self.kernel_size, stride=2, padding=self.kernel_size//2)
        self.layer4 = self.down_layer_2d(self.layer_n * 3 + self.input_channels, self.layer_n * 4, self.kernel_size, stride=2, padding=self.kernel_size//2)

        # 🌟 FIX: Fully Dynamic Architecture. This automatically connects the matrix output correctly
        # The Unet downsamples by a total factor of 8 (three stride=2 layers).
        self.fc = nn.Linear(self.spec_time // 8, 1)
        self.fc2 = nn.Linear(self.spec_freq // 8, 1)

    def down_layer_2d(self, in_channels, out_channels, kernel_size, stride, padding):
        return nn.Sequential(conbr_block_2d(in_channels, out_channels, kernel_size, stride, padding))

    def forward(self, x):
        pool_x1 = self.AvgPool2D1(x)  
        pool_x2 = self.AvgPool2D2(x)  
        out_0 = self.layer1(x)         
        out_1 = self.layer2(out_0)     
        
        x1 = torch.cat([out_1, pool_x1], dim=1)  
        out_2 = self.layer3(x1)         
        
        x2 = torch.cat([out_2, pool_x2], dim=1)  
        x3 = self.layer4(x2)            
        
        # 🌟 FIX: Use safe squeezing to ensure we don't accidentally squeeze the batch size dimension.
        out = self.fc(x3).squeeze(-1)
        out = self.fc2(out).squeeze(-1)
        return None, out


# ==========================================
# 3. PRE-TRAINING (ABlated ISOALIGN)
# ==========================================

# 🚀 Inline replacement for the missing IsoAlign Framework
class IsoAlign(nn.Module):
    def __init__(self, backbone, spect_encoder, FT_encoder, DEVICE, dim, batch_size, args):
        super(IsoAlign, self).__init__()
        self.encoder = backbone       # The Time Encoder
        self.spect_encoder = spect_encoder # The Gabor/Wavelet Encoder
        self.FT_encoder = FT_encoder
        self.DEVICE = DEVICE
        self.args = args
        self.temperature = 0.2 # Temperature scalar for contrastive loss

    def nt_xent_loss(self, z1, z2):
        """Standard Multi-View Contrastive Loss (InfoNCE)"""
        # L2 Normalize the extracted embeddings
        z1 = F.normalize(z1, dim=1)
        z2 = F.normalize(z2, dim=1)
        
        # Calculate cosine similarity matrix
        sim_matrix = torch.matmul(z1, z2.T) / self.temperature
        
        # In contrastive learning, the diagonal represents the positive pairs
        labels = torch.arange(z1.shape[0]).to(self.DEVICE)
        
        # Symmetric Cross-Entropy
        loss_1 = F.cross_entropy(sim_matrix, labels)
        loss_2 = F.cross_entropy(sim_matrix.T, labels)
        return (loss_1 + loss_2) / 2.0

    def forward(self, batch_t, batch_w, batch_f=None):
        # 1. Extract latent representations from the Time branch
        _, feat_t = self.encoder(batch_t)
        
        # 2. Extract latent representations from the Gabor/Wavelet branch
        _, feat_w = self.spect_encoder(batch_w)
        
        # 3. Align them using contrastive loss
        loss = self.nt_xent_loss(feat_t, feat_w)
        return loss

pretrain_loader = DataLoader(SleepEDF_Pretrain_Dataset(split="train"), batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

# Determine shapes dynamically (Only unpacking 2 items now)
_, sample_w = SleepEDF_Pretrain_Dataset(split="train")[0]
SPECT_FREQ = sample_w.shape[1]   
SPECT_TIME = sample_w.shape[2]   

time_encoder = ResNet1D(
    in_channels=1, base_filters=32, kernel_size=5, stride=1, groups=1, 
    n_block=3, n_classes=LATENT_DIM, downsample_gap=2, increasefilter_gap=4, 
    use_do=True, backbone=True, output_dim=LATENT_DIM
).to(DEVICE)

spect_encoder = UNET_2D_simp(
    input_channels=1, output_channels=LATENT_DIM, layer_n=32, 
    spect_freq=SPECT_FREQ, spect_time=SPECT_TIME, kernel_size=3
).to(DEVICE)

# 🚀 THE ABLATION FLAG: Tell IsoAlign to ignore the Fourier branch
class Args: 
    wo_OB = False
    wo_OF = True  # <--- WITHOUT FOURIER = TRUE

# Pass FT_encoder=None
model = IsoAlign(backbone=time_encoder, spect_encoder=spect_encoder, FT_encoder=None, DEVICE=DEVICE, dim=LATENT_DIM, batch_size=BATCH_SIZE, args=Args()).to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

print("\n🚀 Starting Self-Supervised Multi-View Pre-training (Time + Gabor ONLY)...")
best_loss = float('inf')

for epoch in range(PRETRAIN_EPOCHS):
    model.train()
    total_loss = 0
    
    # 🌟 Unpack 2 items instead of 3
    for batch_t, batch_w in pretrain_loader:
        batch_t = batch_t.to(DEVICE)
        
        # 🌟 FIX: Removed .permute(0, 2, 3, 1) entirely. 
        # Conv2D strictly requires [Batch, Channels, Height, Width]
        batch_w = batch_w.to(DEVICE) 
        
        optimizer.zero_grad()
        
        # 🌟 Pass None for the missing Fourier branch
        loss = model(batch_t, batch_w, None) 
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        
    avg_loss = total_loss / len(pretrain_loader)
    print(f"Pretrain Epoch [{epoch+1:02d}/{PRETRAIN_EPOCHS}] | Loss: {avg_loss:.4f}")
    
    if avg_loss < best_loss:
        best_loss = avg_loss
        best_encoder_path = os.path.join(CHECKPOINT_DIR, "time_gabor_best_time_encoder.pth")
        torch.save(model.encoder.state_dict(), best_encoder_path)
        print(f"   🌟 New best loss achieved! Saved encoder to: {best_encoder_path}")
        
    # 💾 Save a comprehensive recovery checkpoint at the end of EVERY epoch
    checkpoint_path = os.path.join(CHECKPOINT_DIR, f"time_gabor_checkpoint_epoch_{epoch+1}.pth")
    torch.save({
        'epoch': epoch + 1,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': avg_loss
    }, checkpoint_path)
    print(f"   💾 Checkpoint saved for epoch {epoch+1} to: {checkpoint_path}")

# ==========================================
# 4. DOWNSTREAM EVALUATION (LINEAR EVAL)
# ==========================================

train_eval_loader = DataLoader(SleepEDF_Evaluation_Dataset(split="train"), batch_size=128, shuffle=True)
val_eval_loader = DataLoader(SleepEDF_Evaluation_Dataset(split="val"), batch_size=128, shuffle=False)
test_eval_loader = DataLoader(SleepEDF_Evaluation_Dataset(split="test"), batch_size=128, shuffle=False)

# Reinitialize backbone and load pre-trained weights
eval_encoder = ResNet1D(
    in_channels=1, base_filters=32, kernel_size=5, stride=1, groups=1, 
    n_block=3, n_classes=LATENT_DIM, downsample_gap=2, increasefilter_gap=4, 
    use_do=False, backbone=True, output_dim=LATENT_DIM
).to(DEVICE)

eval_encoder.load_state_dict(torch.load(os.path.join(CHECKPOINT_DIR, "time_gabor_best_time_encoder.pth"), map_location=DEVICE))
print("\n🔒 Loaded Pre-trained Encoder weights and freezing parameters...")

for param in eval_encoder.parameters():
    param.requires_grad = False

classifier_head = nn.Linear(LATENT_DIM, NUM_CLASSES).to(DEVICE)
criterion = nn.CrossEntropyLoss()
eval_optimizer = optim.Adam(classifier_head.parameters(), lr=1e-2, weight_decay=1e-4)

def evaluate_pipeline(encoder_m, head_m, loader, print_breakdown=False):
    encoder_m.eval(); head_m.eval()
    all_preds, all_targets = [], []
    with torch.no_grad():
        for signals, labels in loader:
            _, features = encoder_m(signals.to(DEVICE))
            preds = torch.argmax(head_m(features), dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(labels.numpy())
            
    acc = accuracy_score(all_targets, all_preds)
    macro_f1 = f1_score(all_targets, all_preds, average='macro')
    weighted_f1 = f1_score(all_targets, all_preds, average='weighted')
    kappa = cohen_kappa_score(all_targets, all_preds)
    
    if print_breakdown:
        CLASS_NAMES = ["Wake (W)", "N1 Stage", "N2 Stage", "N3 Stage", "REM"]
        print("\n📊 DETAILED PERFORMANCE BREAKDOWN (PRE-TRAINED BACKBONE + 1-LAYER LINEAR HEAD):")
        print(f"   ➡️ Test Accuracy:    {acc*100:.2f}%")
        print(f"   ➡️ Cohen's Kappa:     {kappa:.4f}")
        print(f"   ➡️ Macro F1-Score:    {macro_f1:.4f}")
        print(f"   ➡️ Weighted F1-Score: {weighted_f1:.4f}")
        print("   ➡️ Stage-Specific F1-Scores:")
        for name, score in zip(CLASS_NAMES, f1_score(all_targets, all_preds, average=None)):
            print(f"       • {name.ljust(12)}: {score:.4f}")
    return acc, macro_f1

print("\n🏋️ Training Supervised 1-Layer Linear Head...")
best_val_f1 = 0.0

for epoch in range(EVAL_EPOCHS):
    eval_encoder.eval(); classifier_head.train()
    total_loss = 0
    for signals, labels in train_eval_loader:
        eval_optimizer.zero_grad()
        with torch.no_grad():
            _, features = eval_encoder(signals.to(DEVICE))
        loss = criterion(classifier_head(features), labels.to(DEVICE))
        loss.backward()
        eval_optimizer.step()
        total_loss += loss.item()
        
    val_acc, val_f1 = evaluate_pipeline(eval_encoder, classifier_head, val_eval_loader)
    print(f"Eval Epoch [{epoch+1:02d}/{EVAL_EPOCHS}] | Loss: {total_loss/len(train_eval_loader):.4f} | Val Acc: {val_acc*100:.2f}% | Val F1: {val_f1:.4f}")
    
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(classifier_head.state_dict(), os.path.join(CHECKPOINT_DIR, "time_gabor_linear_classifier_head.pth"))

# ==========================================
# 5. FINAL VERIFICATION
# ==========================================
print("\n🔒 Final Deployment Testing on Unseen Participants...")
classifier_head.load_state_dict(torch.load(os.path.join(CHECKPOINT_DIR, "time_gabor_linear_classifier_head.pth"), map_location=DEVICE))
_, _ = evaluate_pipeline(eval_encoder, classifier_head, test_eval_loader, print_breakdown=True)

Executing pipeline on device: cuda:1

✅ Found existing preprocessed dataset at /home/gella.saikrishna/code/Learning-with-FrameProjections/data/sleep_combined.pt. Skipping raw extraction.

🚀 Starting Self-Supervised Multi-View Pre-training (Time + Gabor ONLY)...
Pretrain Epoch [01/40] | Loss: 1.2256
   🌟 New best loss achieved! Saved encoder to: /home/gella.saikrishna/code/Learning-with-FrameProjections/checkpoints/time_gabor_best_time_encoder.pth
   💾 Checkpoint saved for epoch 1 to: /home/gella.saikrishna/code/Learning-with-FrameProjections/checkpoints/time_gabor_checkpoint_epoch_1.pth
